# Part1

dataset_name = Student performance dataset
kaggle_link = https://www.kaggle.com/datasets/devansodariya/student-performance-data

In [1]:
import pandas as pd

df = pd.read_csv('C:/Users/kumar/OneDrive/Desktop/TRY-2/Tasks-Submission/Assignment16/student_data.csv')

# Print shape
print("Shape:", df.shape)

# Column names
print("\nColumns:\n", df.columns)

# First 5 rows
df.head()

Shape: (395, 33)

Columns:
 Index(['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
       'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences', 'G1', 'G2', 'G3'],
      dtype='object')


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [2]:
# Pass = 1, Fail = 0
df["pass"] = (df["G3"] >= 10).astype(int)

In [3]:
X = df.drop(columns=["G3", "pass"])
y = df["pass"]


In [4]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [5]:
cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(exclude="object").columns


In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)


# Task1

In [8]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
svm_linear = Pipeline([
    ("preprocess", preprocessor),
    ("model", SVC(kernel="linear"))
])

svm_linear.fit(X_train, y_train)
y_pred_linear = svm_linear.predict(X_test)

print("Linear SVM Accuracy:", accuracy_score(y_test, y_pred_linear))
svm_rbf = Pipeline([
    ("preprocess", preprocessor),
    ("model", SVC(kernel="rbf"))
])

svm_rbf.fit(X_train, y_train)
y_pred_rbf = svm_rbf.predict(X_test)

print("RBF SVM Accuracy:", accuracy_score(y_test, y_pred_rbf))


Linear SVM Accuracy: 0.8987341772151899
RBF SVM Accuracy: 0.8607594936708861


# Task2

In [10]:
from sklearn.tree import DecisionTreeClassifier
dt_low = Pipeline([
    ("preprocess", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=3, random_state=42))
])

dt_low.fit(X_train, y_train)
print("Low depth train:", dt_low.score(X_train, y_train))
print("Low depth test :", dt_low.score(X_test, y_test))
dt_high = Pipeline([
    ("preprocess", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=20, random_state=42))
])

dt_high.fit(X_train, y_train)
print("High depth train:", dt_high.score(X_train, y_train))
print("High depth test :", dt_high.score(X_test, y_test))


Low depth train: 0.9462025316455697
Low depth test : 0.8860759493670886
High depth train: 1.0
High depth test : 0.8607594936708861


# Part2

# Task3

In [11]:
from sklearn.model_selection import train_test_split
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42
)
model = Pipeline([
    ("preprocess", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=5))
])

model.fit(X_train, y_train)

print("Validation accuracy:", model.score(X_val, y_val))
print("Test accuracy:", model.score(X_test, y_test))


Validation accuracy: 0.8227848101265823
Test accuracy: 0.9113924050632911


# Task4

In [12]:
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(
    model, X, y, cv=5, scoring="accuracy"
)

print("CV scores:", cv_scores)
print("Average CV accuracy:", cv_scores.mean())


CV scores: [0.92405063 0.91139241 0.83544304 0.86075949 0.92405063]
Average CV accuracy: 0.8911392405063292


# Part3

# Task5

In [13]:
from sklearn.ensemble import BaggingClassifier
bagging = Pipeline([
    ("preprocess", preprocessor),
    ("model", BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        n_estimators=50,
        random_state=42
    ))
])

bagging.fit(X_train, y_train)
print("Bagging accuracy:", bagging.score(X_test, y_test))
from sklearn.ensemble import AdaBoostClassifier
adaboost = Pipeline([
    ("preprocess", preprocessor),
    ("model", AdaBoostClassifier(
        n_estimators=50,
        random_state=42
    ))
])

adaboost.fit(X_train, y_train)
print("AdaBoost accuracy:", adaboost.score(X_test, y_test))


Bagging accuracy: 0.8987341772151899
AdaBoost accuracy: 0.9240506329113924


# Task6

In [14]:
from sklearn.ensemble import RandomForestClassifier
rf = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

rf.fit(X_train, y_train)
print("Random Forest accuracy:", rf.score(X_test, y_test))
rf_model = rf.named_steps["model"]
importances = rf_model.feature_importances_

print("Top feature importance values:")
print(importances[:10])


Random Forest accuracy: 0.8987341772151899
Top feature importance values:
[0.02576743 0.01456244 0.01863475 0.01127341 0.01194441 0.03256192
 0.01648682 0.01778386 0.02369563 0.00875808]
